In [1]:
import os
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import random
 
def fijar_semillas(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # opcional, más reproducible pero algo más lento:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
 

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
class Conv1DAutoencoder(nn.Module):
    def __init__(self, input_channels=2, seq_len=61, bottleneck_dim=1):
        super().__init__()

        self.input_channels = input_channels
        self.seq_len = seq_len

        # ============================================================
        # ENCODER CONVOLUCIONAL
        # ============================================================
        self.encoder = nn.Sequential(
            nn.Conv1d(input_channels, 64, kernel_size=7, stride=1, padding=1),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),  # 61 -> 28

            nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=0),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),  # 28 -> 12

            nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=0),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),  # 12 -> 5
        )

        self.flatten = nn.Flatten()

        # ============================================================
        # FC ENCODER (Bottleneck)
        # ============================================================
        self.fc_enc = nn.Sequential(
            nn.Linear(256 * 5, 128),
            nn.ELU(),
            nn.Linear(128, 64),
            nn.ELU(),
            nn.Linear(64, 32),
            nn.ELU(),
            nn.Linear(32, bottleneck_dim),
            nn.ELU()  # Activación ELU en el espacio latente (Tabla 2 del paper)
        )

        # ============================================================
        # FC DECODER (Adaptable dinámicamente)
        # ============================================================
        self.fc_dec = nn.Sequential(
            nn.Linear(bottleneck_dim, 32),
            nn.ELU(),
            nn.Linear(32, 64),
            nn.ELU(),
            nn.Linear(64, 128),
            nn.ELU(),
            nn.Linear(128, input_channels * seq_len),  # 2x61=122 o 4x61=244
        )

    def forward(self, x):
        x_enc = self.encoder(x)                            # (B, 256, 5)
        x_flat = self.flatten(x_enc)                       # (B, 1280)
        z = self.fc_enc(x_flat)                            # (B, bottleneck_dim)
        x_dec = self.fc_dec(z)                             # (B, C * T)
        x_dec = x_dec.view(-1, self.input_channels, self.seq_len) # (B, C, T)
        return x_dec

In [4]:
def entrenar_cae_por_clase(
    X_train,
    model_class,
    cultivo,
    input_channels,
    seq_len=61,
    bottleneck_dim=1,
    batch_size=128,
    epochs=20,
    lr=1e-3,
    device='cpu',
    ruta_modelo=None,      
    verbose=False,         
):
    if not isinstance(X_train, torch.Tensor):
        X_train = torch.as_tensor(X_train, dtype=torch.float32)
 
    loader = DataLoader(TensorDataset(X_train), batch_size=batch_size, shuffle=True)
 
    model = model_class(input_channels=input_channels, seq_len=seq_len,
                        bottleneck_dim=bottleneck_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
 
    hist_train = []
    for epoch in range(epochs):
        model.train()
        loss_epoch = 0.0
        for (x_batch,) in loader:
            x_batch = x_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), x_batch)
            loss.backward()
            optimizer.step()
            loss_epoch += loss.item() * x_batch.size(0)
        loss_epoch /= len(loader.dataset)
        hist_train.append(loss_epoch)
        if verbose:
            print(f"Epoch {epoch+1:02d}/{epochs} - {cultivo} - Train MSE: {loss_epoch:.6f}")
 
    if ruta_modelo is not None:
        os.makedirs(os.path.dirname(ruta_modelo) or ".", exist_ok=True)
        torch.save({'model_state_dict': model.state_dict(), 'loss': hist_train[-1]}, ruta_modelo)
 
    return model, hist_train

def predict_pixels(X, labels, cae_models, batch_size=4096, device='cuda'):
    """
    Calcula las reconstrucciones y el error cuadrático medio (MSE) a nivel de píxel.
    Funciona automáticamente para 2 canales (backscatter) o 4 canales (+ coherencia)
    y soporta etiquetas de texto (strings).
    """
    if isinstance(X, np.ndarray):
        X = torch.as_tensor(X, dtype=torch.float32)
        
    # Asegurar que labels sea un ndarray de NumPy sin convertirlo a torch.Tensor
    if isinstance(labels, torch.Tensor):
        labels_arr = labels.cpu().numpy()
    elif isinstance(labels, list):
        labels_arr = np.array(labels)
    else:
        labels_arr = labels
        
    num_samples = X.shape[0]
    
    # Ordenar las clases de los modelos expertos
    class_ids = sorted(list(cae_models.keys()))
    num_classes = len(class_ids)
    
    # Matriz para almacenar los errores de reconstrucción (N, num_clases)
    mse_matrix = np.zeros((num_samples, num_classes), dtype=np.float32)
    criterion = nn.MSELoss(reduction='none')
    
    for col_idx, c_id in enumerate(class_ids):
        model = cae_models[c_id].to(device)
        model.eval()
        
        col_mse = []
        
        with torch.no_grad():
            for start_idx in range(0, num_samples, batch_size):
                end_idx = min(start_idx + batch_size, num_samples)
                X_batch = X[start_idx:end_idx].to(device)
                
                # Forward pass a través del autoencoder
                X_recon = model(X_batch)
                
                # Promediar el error sobre canales y tiempo
                batch_mse = criterion(X_batch, X_recon).mean(dim=tuple(range(1, X_batch.ndim)))
                col_mse.append(batch_mse.cpu().numpy())
        
        mse_matrix[:, col_idx] = np.concatenate(col_mse)

    # Identificar la clase experta con el menor error de reconstrucción (argmin MSE)
    min_indices = np.argmin(mse_matrix, axis=1)
    class_map = np.array(class_ids)
    candidate_classes = class_map[min_indices]
    
    # Comparación directa entre arreglos de NumPy (funciona con strings e ints)
    is_trustworthy = (candidate_classes == labels_arr)
    is_suspicious = ~is_trustworthy
    
    return {
        'mse_matrix': mse_matrix,
        'candidate_classes': candidate_classes,
        'is_trustworthy': is_trustworthy,
        'is_suspicious': is_suspicious
    }

In [ ]:
"""
Entrenamiento FARMSAR (solo CAE, sin Otsu) sobre los 70 conjuntos infectados:
7 proporciones de ruido x 10 repeticiones, 10 pasadas de purga cada uno.
 
REQUISITO: ejecuta ANTES, en el mismo kernel, las celdas del script de
entrenamiento corregido que definen
    Conv1DAutoencoder, fijar_semillas, entrenar_cae_por_clase (versión SIN
    validación) y predict_pixels.
No las redefinas aquí: la versión antigua de entrenar_cae_por_clase (con
validación, scheduler y selección de mejor época) las sobrescribiría.
"""
# ============================================================
# CONFIGURACIÓN
# ============================================================
dir_base_datasets = "/home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/dataset_ruidoso_farmsar_test_v1"
dir_base_modelos = "/home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/modelos_experimento_farmsar_test_v1"
os.makedirs(dir_base_modelos, exist_ok=True)
 
SPLIT_KEY = "train"
MIN_MUESTRAS_CLASE = 500
noise_levels = [1, 5, 10, 15, 20, 25, 30]
n_repetitions = 10
NUM_PASADAS = 10
EPOCHS = 20
BATCH_SIZE = 128          # el artículo usa 128
LR = 1e-3
BOTTLENECK_DIM = 1
INPUT_CHANNELS = 2
SEQ_LEN = 61
device = 'cuda' if torch.cuda.is_available() else 'cpu'
 
print(f"Dispositivo: {device}")
 
# ============================================================
# BUCLE: 7 proporciones x 10 repeticiones
# ============================================================
for p_pct in noise_levels:
    for rep in range(n_repetitions):
 
        nombre_dataset = f"dataset_noise_{p_pct:02d}pct_rep_{rep:02d}.pt"
        ruta_pt = os.path.join(dir_base_datasets, nombre_dataset)
        if not os.path.exists(ruta_pt):
            print(f"No existe {ruta_pt}, se salta.")
            continue
 
        dir_replica = os.path.join(dir_base_modelos, f"ruido_{p_pct:02d}pct", f"rep_{rep:02d}")
        os.makedirs(dir_replica, exist_ok=True)
 
        # Permite reanudar: si la réplica terminó, no se repite
        marca_fin = os.path.join(dir_replica, "COMPLETO.txt")
        if os.path.exists(marca_fin):
            print(f"Ruido {p_pct}% rep {rep}: ya completo, se salta.")
            continue
 
        print("\n" + "=" * 70)
        print(f"Ruido {p_pct}% | réplica {rep + 1}/{n_repetitions}")
        print("=" * 70)
 
        # Semilla distinta y reproducible por experimento
        fijar_semillas(1000 * p_pct + rep)
 
        # ---- Cargar conjunto infectado ----
        data = torch.load(ruta_pt, map_location="cpu", weights_only=False)
        assert SPLIT_KEY in data, f"{nombre_dataset} no tiene la clave '{SPLIT_KEY}': {list(data.keys())}"
        datos_inf = data[SPLIT_KEY]
        del data
 
        class_ids = sorted(datos_inf.keys())
        list_X, list_labels = [], []
        for crop_id in class_ids:
            X_crop = torch.as_tensor(datos_inf[crop_id]["X"], dtype=torch.float32)
            if X_crop.ndim == 2:
                X_crop = X_crop.reshape(-1, INPUT_CHANNELS, SEQ_LEN)
            list_X.append(X_crop)
            list_labels.extend([crop_id] * len(X_crop))
 
        X_global = torch.cat(list_X, dim=0)
        labels_global = np.array(list_labels)
        del list_X, datos_inf
        active_mask = np.ones(len(labels_global), dtype=bool)
 
        # ---- Reanudar dentro de la réplica ----
        # Cada pasada entrena desde cero con los píxeles activos, así que basta
        # la máscara guardada tras la purga p para empezar en la pasada p+1.
        pasada_ini = 1
        for p_prev in range(NUM_PASADAS - 1, 0, -1):
            f_mask = os.path.join(dir_replica, f"active_mask_tras_pasada_{p_prev}.npy")
            if not os.path.exists(f_mask):
                continue
            try:
                m_prev = np.load(f_mask)
            except Exception:
                continue                       # archivo corrupto (corte a mitad): probar la anterior
            if len(m_prev) == len(labels_global):
                active_mask = m_prev.astype(bool)
                pasada_ini = p_prev + 1
                print(f"  Reanudando en la pasada {pasada_ini} (máscara de la pasada {p_prev}).")
                break
 
        # ---- 10 pasadas de purga ----
        for pasada in range(pasada_ini, NUM_PASADAS + 1):
            es_ultima = pasada == NUM_PASADAS
            modelos_pasada = {}
 
            for crop_id in class_ids:
                m = (labels_global == crop_id) & active_mask
                if m.sum() < MIN_MUESTRAS_CLASE:
                    print(f"  [{crop_id}] solo {m.sum()} muestras activas "
                        f"(< {MIN_MUESTRAS_CLASE}), se congela sin purgar.")
                    continue
 
                # Solo se guardan a disco los modelos de la última pasada
                ruta = (os.path.join(dir_replica, f"cae_pasada_{pasada}_crop_{crop_id}.pth")
                        if es_ultima else None)
 
                modelo, _ = entrenar_cae_por_clase(
                    X_global[torch.as_tensor(m)],
                    Conv1DAutoencoder,
                    crop_id,
                    input_channels=INPUT_CHANNELS,
                    seq_len=SEQ_LEN,
                    bottleneck_dim=BOTTLENECK_DIM,
                    batch_size=BATCH_SIZE,
                    epochs=EPOCHS,
                    lr=LR,
                    device=device,
                    ruta_modelo=ruta,
                    verbose=False,
                )
                modelos_pasada[crop_id] = modelo
 
            if not es_ultima:
                # Se evalúan solo los píxeles aún activos; los purgados no vuelven
                idx = np.where(active_mask)[0]
                res = predict_pixels(
                    X_global[torch.as_tensor(idx)],
                    labels_global[idx],
                    modelos_pasada,
                    batch_size=4096,
                    device=device,
                )
                active_mask[idx[~res["is_trustworthy"]]] = False
                print(f"  Pasada {pasada}: activos {active_mask.sum()} / {len(labels_global)}")
                # Guardado atómico: escribe a temporal y renombra
                f_mask = os.path.join(dir_replica, f"active_mask_tras_pasada_{pasada}.npy")
                np.save(f_mask + ".tmp.npy", active_mask)
                os.replace(f_mask + ".tmp.npy", f_mask)
 
        with open(marca_fin, "w") as f:
            f.write("COMPLETO")
        del X_global, modelos_pasada
        if device == 'cuda':
            torch.cuda.empty_cache()
 
print("\nEjecución completada.")

#SI soy yo

Dispositivo: cuda

Ruido 1% | réplica 1/10
  Pasada 1: activos 387150 / 413496
  Pasada 2: activos 371081 / 413496
  Pasada 3: activos 368050 / 413496
  Pasada 4: activos 365950 / 413496
  Pasada 5: activos 354857 / 413496
  Pasada 6: activos 353549 / 413496
  Pasada 7: activos 352484 / 413496
  Pasada 8: activos 347280 / 413496
  Pasada 9: activos 343120 / 413496

Ruido 1% | réplica 2/10
  Pasada 1: activos 372496 / 413496
  Pasada 2: activos 357557 / 413496
  Pasada 3: activos 349301 / 413496
  Pasada 4: activos 346035 / 413496
  Pasada 5: activos 338330 / 413496
  Pasada 6: activos 337004 / 413496
  Pasada 7: activos 334109 / 413496
  Pasada 8: activos 333373 / 413496
  Pasada 9: activos 332985 / 413496

Ruido 1% | réplica 3/10
  Pasada 1: activos 368633 / 413496
  Pasada 2: activos 334987 / 413496
  Pasada 3: activos 332255 / 413496
  Pasada 4: activos 330984 / 413496
  Pasada 5: activos 329406 / 413496
  Pasada 6: activos 328280 / 413496
  Pasada 7: activos 326554 / 413496
  Pasad

In [ ]:
dir_base_modelos = "/home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento y fiables en train/modelos_experimento_farmsar_test_v1"
for p in [1, 5, 10, 15, 20, 25, 30]:
    hechas = [r for r in range(10) if os.path.exists(
        os.path.join(dir_base_modelos, f"ruido_{p:02d}pct", f"rep_{r:02d}", "COMPLETO.txt"))]
    print(f"{p:>2} %: réplicas completas {hechas}")

 1 %: réplicas completas [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
 5 %: réplicas completas [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
10 %: réplicas completas [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
15 %: réplicas completas [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
20 %: réplicas completas [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
25 %: réplicas completas [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
30 %: réplicas completas [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
